In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')

# 1. VERİLERİ GÜVENLİ YÜKLEME VE TEMİZLEME
matches_df = pd.read_csv('data/processed/matches.csv')
maps_df = pd.read_csv('data/processed/maps.csv')
player_df = pd.read_csv('data/processed/player_stats.csv')

# Kolonları yazdırıp şemayı teyit edelim
print("MATCHES columns:", matches_df.columns.tolist())
print("MAPS columns:", maps_df.columns.tolist())
print("PLAYERS columns:", player_df.columns.tolist())

# Şemaya göre temizlik yap
if 'match_id' in matches_df.columns:
    matches_df['match_id'] = matches_df['match_id'].astype(str).str.extract(r'(\d+)')[0]

if 'match_id' in maps_df.columns:
    maps_df['match_id'] = maps_df['match_id'].astype(str).str.extract(r'(\d+)')[0]

if 'map_id' in maps_df.columns:
    maps_df['map_id'] = maps_df['map_id'].astype(str).str.strip()

if 'map_id' in player_df.columns:
    player_df['map_id'] = player_df['map_id'].astype(str).str.strip()

# Harita ismi temizliği
maps_df['map_name'] = maps_df['map_name'].astype(str).str.lower().str.strip()
valid_maps = ['ascent', 'bind', 'haven', 'split', 'lotus', 'sunset', 'abyss', 'icebox', 'fracture', 'breeze', 'pearl']
maps_df['map_name'] = maps_df['map_name'].apply(lambda x: next((m for m in valid_maps if m in x), np.nan))
maps_df = maps_df.dropna(subset=['map_name'])

# BİRLEŞTİRME: Matches -> Maps
df_train = pd.merge(matches_df, maps_df, on='match_id', how='inner')

# Target'ı belirle
df_train['team_a_won_map'] = (df_train['team_a_score'] > df_train['team_b_score']).astype(int)

# BİRLEŞTİRME: df_train -> Players (map_id üzerinden)
player_df['agent'] = player_df['agent'].astype(str).str.split(',').str[0].str.strip()
team_agents = pd.get_dummies(player_df[['map_id', 'team', 'agent']], columns=['agent'], prefix='', prefix_sep='').groupby(['map_id', 'team']).max().reset_index()

team_a = team_agents[team_agents['team'] == 'team_a'].drop('team', axis=1).rename(columns=lambda x: x if x == 'map_id' else f"{x}_t1")
team_b = team_agents[team_agents['team'] == 'team_b'].drop('team', axis=1).rename(columns=lambda x: x if x == 'map_id' else f"{x}_t2")

df_train = df_train.merge(team_a, on='map_id', how='left').merge(team_b, on='map_id', how='left').fillna(0)

print(f"Final Satır Sayısı: {len(df_train)}")
assert len(df_train) > 0, "Eğitim verisi boş! Lütfen CSV dosyalarını kontrol edin."

# 3. HARİTALARI İŞLEME (ONE-HOT ENCODING)
map_dummies = pd.get_dummies(df_train['map_name'], prefix='map')
df_train = pd.concat([df_train, map_dummies], axis=1)

# 4. SÜTUNLARI AYIRMA
map_cols = list(map_dummies.columns)
t1_agent_cols = [col for col in df_train.columns if col.endswith('_t1') and col != 'team_t1']
t2_agent_cols = [col for col in df_train.columns if col.endswith('_t2') and col != 'team_t2']

# 5. SİMETRİK VERİ ÜRETİMİ
df_normal = df_train[map_cols + t1_agent_cols + t2_agent_cols].copy()
df_normal['target'] = df_train['team_a_won_map']

df_swapped = df_train[map_cols + t2_agent_cols + t1_agent_cols].copy()
df_swapped.columns = map_cols + t1_agent_cols + t2_agent_cols
df_swapped['target'] = 1 - df_train['team_a_won_map'] 

df_symmetric = pd.concat([df_normal, df_swapped], ignore_index=True)
df_symmetric = df_symmetric.fillna(0)

# Bool'ları integer'a çevir
bool_cols = df_symmetric.select_dtypes(include=['bool']).columns
df_symmetric[bool_cols] = df_symmetric[bool_cols].astype(int)

X = df_symmetric.drop('target', axis=1)
y = df_symmetric['target']

# 6. EĞİTİM VE TEST
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestClassifier(
    n_estimators=200, 
    max_depth=15, 
    min_samples_split=5, 
    random_state=42
)
model.fit(X_train, y_train)

print(f"🚀 Simetrik Model Doğruluk Oranı: %{accuracy_score(y_test, model.predict(X_test)) * 100:.2f}")

joblib.dump(model, 'valorant_rf_model.pkl')
joblib.dump(list(X.columns), 'model_columns.pkl')
print("✅ Model ve Sütunlar Kaydedildi!")

# 7. HARİTAYA ÖZEL AJAN İSTATİSTİKLERİ
map_agent_stats = {}
for m_col in map_cols:
    map_name = m_col.replace('map_', '')
    map_data = df_train[df_train[m_col] == 1]
    
    agent_win_rates = {}
    for agent_col in t1_agent_cols:
        agent_name = agent_col.replace('_t1', '')
        agent_played = map_data[map_data[agent_col] == 1]
        
        if len(agent_played) > 3: 
            win_rate = agent_played['team_a_won_map'].mean() * 100
            agent_win_rates[agent_name] = round(win_rate, 1)
            
    sorted_agents = sorted(agent_win_rates.items(), key=lambda item: item[1], reverse=True)[:5]
    map_agent_stats[map_name] = sorted_agents

joblib.dump(map_agent_stats, 'map_agent_stats.pkl')
print("✅ Haritaya Özel Ajan İstatistikleri (map_agent_stats.pkl) Kaydedildi!")

KeyError: 'match_id'